# 01 — Feature Extraction (Methods)

Extracts windowed EEG features for BOCPD. Produces **log bandpower** per trial × channel × band (δ, θ, α, β, γ). Outputs saved to `artifacts/data/` for use by BOCPD notebooks.

Reference: `refrences/deep-research-report.md` — run BOCPD on log bandpower (Gaussian conjugate).

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.signal import welch

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else (Path.cwd() / ".." / "..").resolve()
DATA_ROOT = ROOT / "data"
ARTIFACTS_DATA = ROOT / "artifacts" / "data"
ARTIFACTS_DATA.mkdir(parents=True, exist_ok=True)
PARTICIPANTS = [f"sub-{i:02d}" for i in range(1, 11)]
DATE_TAG = "2026-02-27"
FS = 100

In [3]:
def load_preprocessed(path):
    d = np.load(path, allow_pickle=True).item()
    return d["preprocessed_eeg_data"], d["ch_names"], d["times"]

BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 50),
}

def bandpower_from_psd(freqs, psd, low, high):
    idx = (freqs >= low) & (freqs <= high)
    return np.trapz(psd[idx], freqs[idx])

def compute_bandpower(x, fs=100):
    """x: 1D array (one trial, one channel). Returns dict of band -> power."""
    f, psd = welch(x, fs=fs, nperseg=min(64, len(x)))
    return {b: bandpower_from_psd(f, psd, lo, hi) for b, (lo, hi) in BANDS.items()}

## Extract log bandpower per trial

For each participant: train (conditions × 4 reps × 17 ch × 100 time) and test (200 cond × 80 reps × 17 ch × 100 time). Output shape per split: (conditions, reps, channels, bands).

In [ ]:
def extract_log_bandpower(X, ch_names, bands_dict):
    """
    X: (conditions, reps, channels, timepoints)
    Returns: (conditions, reps, channels, n_bands) log bandpower array.
    Vectorized: reshape to (n_trials, n_time), welch on all, then bandpower.
    """
    n_cond, n_rep, n_ch, n_time = X.shape
    band_names = list(bands_dict.keys())
    n_bands = len(band_names)
    n_trials = n_cond * n_rep * n_ch
    x_flat = X.reshape(n_trials, n_time)
    f, psd = welch(x_flat, fs=FS, nperseg=min(64, n_time), axis=1)
    out_flat = np.zeros((n_trials, n_bands), dtype=np.float32)
    for i, (b, (lo, hi)) in enumerate(bands_dict.items()):
        idx = (f >= lo) & (f <= hi)
        out_flat[:, i] = np.trapz(psd[:, idx], f[idx], axis=1)
    out_flat = np.log(out_flat + 1e-10)
    return out_flat.reshape(n_cond, n_rep, n_ch, n_bands)

In [ ]:
out_dir = ARTIFACTS_DATA / f"features__log_bandpower__{DATE_TAG}"
out_dir.mkdir(parents=True, exist_ok=True)

for sub in PARTICIPANTS:
    p_train = DATA_ROOT / sub / "preprocessed_eeg_training.npy"
    p_test = DATA_ROOT / sub / "preprocessed_eeg_test.npy"
    if not p_train.exists() or not p_test.exists():
        print(f"Skip {sub}: no data")
        continue
    X_train, ch_names, times = load_preprocessed(p_train)
    X_test, _, _ = load_preprocessed(p_test)
    lbp_train = extract_log_bandpower(X_train, ch_names, BANDS)
    lbp_test = extract_log_bandpower(X_test, ch_names, BANDS)
    np.savez_compressed(
        out_dir / f"{sub}.npz",
        train=lbp_train,
        test=lbp_test,
        ch_names=np.array(ch_names),
        bands=np.array(list(BANDS.keys())),
    )
    print(f"{sub}: train {lbp_train.shape}, test {lbp_test.shape}")

## Summary table (for results.md)

In [ ]:
rows = []
for sub in PARTICIPANTS:
    p = out_dir / f"{sub}.npz"
    if not p.exists():
        continue
    d = np.load(p, allow_pickle=True)
    tr = d["train"]
    te = d["test"]
    rows.append({
        "participant": sub,
        "train_shape": str(tr.shape),
        "test_shape": str(te.shape),
        "train_mean": float(tr.mean()),
        "train_std": float(tr.std()),
        "test_mean": float(te.mean()),
        "test_std": float(te.std()),
    })

summary_df = pd.DataFrame(rows)
summary_path = ROOT / "artifacts" / "tables" / f"features__log_bandpower_summary__{DATE_TAG}.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
print(summary_df.to_string())